### Connecting to an API/Pulling in the Data and Cleaning/Formatting

In [1]:
# import libraries
import pandas as pd
import urllib.request, urllib.parse, urllib.error
import json
import requests
import kaggle
import os
from kaggle.api.kaggle_api_extended import KaggleApi

In [2]:
# 7.4 use with open to call json file with API key
with open('Kaggle.json') as f:
    keys = json.load(f)
    kaggle = keys['maureencasey']

In [3]:
# Ensure the Kaggle API credentials file is in the correct location
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)

In [4]:
# If you have the kaggle.json file in your current directory, move it to the ~/.kaggle directory
if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    import shutil
    shutil.move('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))

In [5]:
# Set the Kaggle configuration directory
os.environ['KAGGLE_CONFIG_DIR'] = os.path.expanduser('~/.kaggle')

In [6]:
# Initialize and authenticate the Kaggle API
api = KaggleApi()
api.authenticate()

In [7]:
# Replace 'census/us-population-by-zip-code' with the dataset you are interested in
dataset = 'census/us-population-by-zip-code'

In [8]:
# Download the dataset
api.dataset_download_files(dataset, path='.', unzip=True)

Dataset URL: https://www.kaggle.com/datasets/census/us-population-by-zip-code


In [9]:
# Assuming the relevant file is named 'us-population-by-zip-code.csv'
csv_file = "Data\population_by_zip_2010.csv"

In [10]:
# Could not find any documentation on Kaggle to make changes directly to JSON source, so I downloaded and CSV and converted to JSON
# Read the CSV file into a DataFrame
df = pd.read_csv(csv_file)

In [11]:
# Print the first few rows of the DataFrame
print(df.head())

   population  minimum_age  maximum_age  gender  zipcode          geo_id
0          50         30.0         34.0  female    61747  8600000US61747
1           5         85.0          NaN    male    64120  8600000US64120
2        1389         30.0         34.0    male    95117  8600000US95117
3         231         60.0         61.0  female    74074  8600000US74074
4          56          0.0          4.0  female    58042  8600000US58042


In [12]:
# Save the DataFrame as a JSON file with no index numbers
json_file = 'Data\population_by_zip_2010.json'
df.to_json(json_file, orient='records', lines=True, index=False)

In [13]:
print(f"\nJSON file saved as: {json_file}")


JSON file saved as: Data\population_by_zip_2010.json


## 1 Transformation - remove some columns

In [14]:
# Specify columns to remove
columns_to_remove = ['minimum_age', 'maximum_age']

In [15]:
# Remove specific columns
df = df.drop(columns=columns_to_remove)

In [16]:
print(df.head())

   population  gender  zipcode          geo_id
0          50  female    61747  8600000US61747
1           5    male    64120  8600000US64120
2        1389    male    95117  8600000US95117
3         231  female    74074  8600000US74074
4          56  female    58042  8600000US58042


## 2 Transformation - change order of columns

In [17]:
# Change the order of the columns
desired_order = ['zipcode', 'geo_id', 'population', 'gender']  # Adjust to your desired order
df = df[desired_order]

In [18]:
print(df.head())

   zipcode          geo_id  population  gender
0    61747  8600000US61747          50  female
1    64120  8600000US64120           5    male
2    95117  8600000US95117        1389    male
3    74074  8600000US74074         231  female
4    58042  8600000US58042          56  female


## 3 Transformation - remove index column

In [19]:
# print original data types
print(df.dtypes)

zipcode        int64
geo_id        object
population     int64
gender        object
dtype: object


## 4 Transformation - change header names

In [20]:
# Rename specific columns
df = df.rename(columns={
    'zipcode': 'ZipCode',
    'population': 'PopulationCensus'
})

## 5 Transformation - change data types of ZipCode column to match with other data sources

In [21]:
# change data types
df = df.astype({
    'ZipCode': 'object'
})

In [22]:
# print new data types
print(df.dtypes)

ZipCode             object
geo_id              object
PopulationCensus     int64
gender              object
dtype: object


## 6 Transformation - grouping

In [23]:
# show how many less unique values exist in dataframe
# how many responses for each gender?
gender_count = df.groupby("gender")["gender"].count()
gender_count.sort_values(ascending = False, inplace=True)
gender_count

gender
female    794856
male      794856
Name: gender, dtype: int64

## 7 Transformations - order by ZipCode, then create herarchy

In [24]:
# Order by 'zipcode'
df_sorted = df.sort_values(by='ZipCode')

In [25]:
# create a hierarchical index
df_hier = df.set_index(['ZipCode', 'PopulationCensus'])

In [26]:
print(df_hier)

                                  geo_id  gender
ZipCode PopulationCensus                        
61747   50                8600000US61747  female
64120   5                 8600000US64120    male
95117   1389              8600000US95117    male
74074   231               8600000US74074  female
58042   56                8600000US58042  female
...                                  ...     ...
28640   66                8600000US28640  female
98604   791               8600000US98604    male
29545   55                8600000US29545  female
45319   10                8600000US45319  female
71032   65                8600000US71032  female

[1622831 rows x 2 columns]


In [27]:
# Group by 'ZipCode' and sum the 'population'
df_grouped = df.groupby('ZipCode', as_index=False)['PopulationCensus'].sum()

In [28]:
print(df_grouped.head())

  ZipCode  PopulationCensus
0     602            124560
1     603            164067
2     606             19845
3     610             87048
4     612            201030


In [29]:
df['ZipCode'] = df['ZipCode'].astype(str).str.zfill(5)

In [30]:
print(df_grouped.head())

  ZipCode  PopulationCensus
0     602            124560
1     603            164067
2     606             19845
3     610             87048
4     612            201030


In [31]:
# create csv for Milestone 5
csv_file = 'Census_API.csv'
df.to_csv(csv_file, index=False)
print(f"\nCSV file saved as: {csv_file}")


CSV file saved as: Census_API.csv


## Ethical implications of data wrangling specific to your datasource and the steps you completed answering the following questions:

### Data was cleaned up so that it will match future data sources. 
### Minimum_age and Maximum_age fields were removed. Gender was also removed in order to get a summary population for each ZipCode. 
### None of the data was assumed. 
### There are no legal or regulatory issues with this project. 
### Permissions were granted on Kaggle site page to use the data.
### Possible risks include, if there were null values in the grouping, then there was a possibilty that the groupings could be totaled incorrectly.### No assumpion wsere made in cleaning or transforming the data. 
### Kaggle was used and per the site, data can be used for educational purposes.
### It was originally sourced from .gov sites. The data waas acquired in an ethical way.
### A possible ethical implication could be if data isn't anonymized, but all data is summary data that isn't personally identifiable at this time

In [32]:
df

,ZipCode,geo_id,PopulationCensus,gender
0,61747,8600000US61747,50,female
1,64120,8600000US64120,5,male
2,95117,8600000US95117,1389,male
3,74074,8600000US74074,231,female
4,58042,8600000US58042,56,female
...,...,...,...,...
1622826,28640,8600000US28640,66,female
1622827,98604,8600000US98604,791,male
1622828,29545,8600000US29545,55,female
1622829,45319,8600000US45319,10,female


In [33]:
df_grouped

,ZipCode,PopulationCensus
0,602,124560
1,603,164067
2,606,19845
3,610,87048
4,612,201030
...,...,...
33114,99923,261
33115,99925,2457
33116,99926,4380
33117,99927,282
